In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import random
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler

from typing import List, Optional, Sequence, Tuple, Dict
from itertools import combinations


from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import timm
from tqdm import tqdm

# Preprocessing

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

SEED = 43

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)


data_dir            = "/path/to/ISIC_DATA"
train_dir            = "/path/to/ISIC_DATA/ISIC_2019_Training_Input"
test_dir             = "/path/to/ISIC_DATA/ISIC_2019_Test_Input"
train_truth_file     = "ISIC_2019_Training_GroundTruth.csv"
test_truth_file      = "ISIC_2019_Test_GroundTruth.csv"
train_metadata_file  = "ISIC_2019_Training_Metadata.csv"


# -------------------------------------------------------------- classes
CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
NUM_CLASSES = len(CLASS_NAMES)
UNK_LABEL = NUM_CLASSES

# ----------------------------------------------------------------- data
IMG_SIZE = 384
BATCH_SIZE = 32
NUM_WORKERS = 4

# ---------------------------------------------------------------- model
ARCH = "tf_efficientnetv2_m.in21k_ft_in1k"
DROPOUT = 0.4
DROP_PATH_RATE = 0.2

# ------------------------------------------------------------- training
NUM_EPOCHS = 35
LR = 1e-4
WEIGHT_DECAY = 0.01
GAMMA = 2.0



In [ ]:
train_gt = pd.read_csv(os.path.join(data_dir, train_truth_file))
test_gt  = pd.read_csv(os.path.join(data_dir, test_truth_file))

# Create single_label for training (exclude image & UNK)
train_labels = train_gt.iloc[:, 1:9]
train_gt['single_label'] = train_labels.values.argmax(axis=1)

# For test (include UNK)
test_labels = test_gt.iloc[:, 1:10]
test_gt['single_label'] = test_labels.values.argmax(axis=1)


meta = pd.read_csv(os.path.join(data_dir, train_metadata_file))

# Merge lesion_id from metadata into train_gt (left join, keyed on image name)
train_gt = train_gt.merge(meta[['image', 'lesion_id']], on='image', how='left')
train_gt['group_id'] = train_gt['lesion_id'].fillna(train_gt['image'])

# Stratified GROUP Train/Val Split

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(
    sgkf.split(train_gt, train_gt['single_label'], groups=train_gt['group_id'])
)

train_df = train_gt.iloc[train_idx].reset_index(drop=True).drop(columns=['group_id', 'lesion_id'])
val_df   = train_gt.iloc[val_idx].reset_index(drop=True).drop(columns=['group_id', 'lesion_id'])

train_groups = set(train_gt.loc[train_idx, 'group_id'])
val_groups   = set(train_gt.loc[val_idx, 'group_id'])
overlap = train_groups & val_groups
print(f"Overlapping groups between train/val: {len(overlap)}")

In [ ]:
class CenterSquareCrop:
    def __call__(self, img):
        w, h = img.size
        m = min(w, h)
        left, top = (w-m)//2, (h-m)//2
        return img.crop((left, top, left+m, top+m))


IMAGENET_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STD = [0.5, 0.5, 0.5]

IMAGENET_MEAN_FILL = tuple(
    round(x * 255) for x in IMAGENET_MEAN
)

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    CenterSquareCrop(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees = 30, translate=(0.01,0.01), scale=(1.0,1.05), fill=IMAGENET_MEAN_FILL),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    CenterSquareCrop(),
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)
])


class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        name = self.df.iloc[idx]['image'] + '.jpg'
        img = Image.open(os.path.join(self.img_dir, name))
        label = int(self.df.iloc[idx]['single_label'])
        if self.transform:
            img = self.transform(img)
        return img, label


train_ds = ISICDataset(train_df, train_dir, train_transform)
val_ds   = ISICDataset(val_df,   train_dir, val_transform)
test_ds  = ISICDataset(test_gt,  test_dir,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, worker_init_fn=seed_worker, generator=g,)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,num_workers=4, pin_memory=True)

## Training Routine

### Model

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        loss = ((1 - p_t) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    all_preds = []
    all_labels = []

    for inputs, targets in tqdm(loader, desc='Train'):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():
            logits = model(inputs)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Loss
        total_loss += loss.item() * targets.size(0)

        # Predictions / metrics
        with torch.no_grad():
            preds = logits.argmax(dim=1)

            total_correct += (preds == targets).sum().item()
            total_samples += targets.size(0)

            all_preds.append(preds.detach().cpu())
            all_labels.append(targets.detach().cpu())

    # Epoch metrics
    avg_loss = total_loss / total_samples
    avg_acc = 100.0 * total_correct / total_samples

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    macro_f1 = f1_score(
        all_labels.numpy(),
        all_preds.numpy(),
        average='macro'
    )

    return avg_acc, avg_loss, macro_f1


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    for inputs, targets in tqdm(loader, desc='Eval'):
        inputs, targets = inputs.to(device), targets.to(device)
        logits = model(inputs)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(targets.cpu())

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    acc = 100. * (all_preds == all_labels).float().mean().item()
    macro_f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average='macro')
    return acc, macro_f1


In [ ]:
model = timm.create_model(
    ARCH,
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT,          
    drop_path_rate=DROP_PATH_RATE,
).to(device)

criterion = FocalLoss(gamma=GAMMA, reduction='mean')
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler()

In [ ]:
best_val_f1 = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    train_acc, train_loss, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        device
    )
    val_acc, val_f1 = evaluate(model, val_loader, device)

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Train Macro F1: {train_f1:.4f}"
    )

    print(
        f"Val Acc: {val_acc:.2f}% | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'baseline_checkpoint.pth')
        print(f"Saved best model (F1={best_val_f1:.4f})")

print("Training complete.")

# Test

In [ ]:
model.load_state_dict(torch.load('baseline_checkpoint.pth', weights_only=True))
model.to(device)

In [ ]:
@torch.no_grad()
def get_predictions_and_uncertainty(
    model,
    inputs: torch.Tensor,
):

    logits = model(inputs)

    probs = torch.softmax(
        logits,
        dim=1,
    )

    preds = probs.argmax(dim=1)

    uncertainty = (
        1.0 - probs.max(dim=1).values
    )

    return preds, uncertainty

In [ ]:
def compute_auroc(labels: np.ndarray, scores: np.ndarray, score_name: str, task_name: str):

    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())

    if n_pos == 0 or n_neg == 0:
        print(f"[{score_name}] Cannot compute {task_name} AUROC: n_positive={n_pos}, n_negative={n_neg}")
        return None

    auroc = roc_auc_score(labels, scores)
    print(f"[{score_name}] {task_name} AUROC: {auroc:.4f} (n_negative={n_neg}, n_positive={n_pos})")
    return auroc

In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    dataloader,
    device,
    num_classes=8,
    unk_label=8,
):
    model.eval()

    all_preds = []
    all_targets = []
    all_uncertainty = []

    for inputs, targets in tqdm(
        dataloader,
        desc="Evaluating",
        ncols=100,
    ):
        inputs = inputs.to(device)
        targets = targets.to(device)

        preds, uncertainty = (
            get_predictions_and_uncertainty(
                model,
                inputs,
            )
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_targets.extend(
            targets.cpu().numpy()
        )

        all_uncertainty.extend(
            uncertainty.cpu().numpy()
        )

    all_preds = np.asarray(all_preds)
    all_targets = np.asarray(all_targets)
    all_uncertainty = np.asarray(
        all_uncertainty
    )

    # --------------------------------------------------
    # ID classification metrics

    id_mask = all_targets != unk_label

    preds_id = all_preds[id_mask]
    targets_id = all_targets[id_mask]
    uncertainty_id = all_uncertainty[id_mask]

    macro_f1 = f1_score(
        targets_id,
        preds_id,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc = balanced_accuracy_score(
        targets_id,
        preds_id,
    )

    # --------------------------------------------------
    # UNK detection

    is_unk = (
        all_targets == unk_label
    ).astype(int)

    unk_auroc = compute_auroc(
        is_unk,
        all_uncertainty,
        "1 - MSP",
        "UNK-detection",
    )


    return (
        macro_f1,
        balanced_acc,
        unk_auroc,
    )

In [ ]:
(
    test_macro_f1,
    test_balanced_acc,
    test_unk_auroc,
) = evaluate_model(
    model,
    test_loader,
    device,
)

print(f"Macro F1:                         {test_macro_f1:.4f}")
print(f"Balanced Accuracy:                {test_balanced_acc:.4f}")
print(f"UNK-detection AUROC:              {test_unk_auroc:.4f}")

In [ ]:
@torch.no_grad()
def collect_test_predictions(
    model,
    dataloader,
    device,
):
    model.eval()

    all_preds = []
    all_targets = []
    all_uncertainty = []

    for inputs, targets in tqdm(
        dataloader,
        desc="Collecting test predictions",
        ncols=100,
    ):
        inputs = inputs.to(device)
        targets = targets.to(device)

        preds, uncertainty = (
            get_predictions_and_uncertainty(
                model,
                inputs,
            )
        )

        all_preds.extend(
            preds.cpu().numpy()
        )
        all_targets.extend(
            targets.cpu().numpy()
        )
        all_uncertainty.extend(
            uncertainty.cpu().numpy()
        )

    return (
        np.asarray(all_preds),
        np.asarray(all_targets),
        np.asarray(all_uncertainty),
    )

def evaluate_id_after_pruning(
    preds,
    targets,
    uncertainty,
    prune_frac=0.10,
    num_classes=8,
    unk_label=8,
):

    id_mask = targets != unk_label

    preds_id = preds[id_mask]
    targets_id = targets[id_mask]
    uncertainty_id = uncertainty[id_mask]

    n_id = len(targets_id)
    n_prune = int(round(n_id * prune_frac))
    n_keep = n_id - n_prune

    # Ascending uncertainty:
    # least uncertain samples are retained.
    order = np.argsort(uncertainty_id)
    keep_idx = order[:n_keep]

    kept_preds = preds_id[keep_idx]
    kept_targets = targets_id[keep_idx]

    macro_f1_before = f1_score(
        targets_id,
        preds_id,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc_before = balanced_accuracy_score(
        targets_id,
        preds_id,
    )

    macro_f1_after = f1_score(
        kept_targets,
        kept_preds,
        average="macro",
        labels=list(range(num_classes)),
    )

    balanced_acc_after = balanced_accuracy_score(
        kept_targets,
        kept_preds,
    )

    print(
        f"[ID-only: reject "
        f"{prune_frac:.0%} most uncertain]"
    )

    print(
        f"  N ID: {n_id} | "
        f"N retained: {n_keep} | "
        f"N rejected: {n_prune}"
    )

    print(
        f"  Macro F1:          "
        f"{macro_f1_before:.4f} -> "
        f"{macro_f1_after:.4f}"
    )

    print(
        f"  Balanced Accuracy: "
        f"{balanced_acc_before:.4f} -> "
        f"{balanced_acc_after:.4f}"
    )

    return {
        "n_id": n_id,
        "n_retained": n_keep,
        "n_rejected": n_prune,
        "macro_f1_before": macro_f1_before,
        "macro_f1_after": macro_f1_after,
        "balanced_acc_before": balanced_acc_before,
        "balanced_acc_after": balanced_acc_after,
    }

def evaluate_full_pruning_composition(
    targets,
    uncertainty,
    prune_frac=0.10,
    unk_label=8,
):

    n_total = len(targets)
    n_prune = int(round(n_total * prune_frac))

    # Reject the most uncertain samples
    rejected_idx = np.argsort(
        -uncertainty
    )[:n_prune]

    is_unk = targets == unk_label
    n_unk = int(is_unk.sum())

    n_rejected_unk = int(
        is_unk[rejected_idx].sum()
    )

    unk_caught = (
        n_rejected_unk / n_unk
        if n_unk > 0
        else np.nan
    )

    print(
        f"[Full test set: reject "
        f"{prune_frac:.0%} most uncertain]"
    )

    print(
        f"  N total: {n_total} | "
        f"N rejected: {n_prune}"
    )

    print(
        f"  UNK caught: "
        f"{n_rejected_unk}/{n_unk} "
        f"({unk_caught:.2%})"
    )

    return {
        "n_total": n_total,
        "n_rejected": n_prune,
        "unk_caught": unk_caught,
        "n_rejected_unk": n_rejected_unk,
    }



In [ ]:
# Collect predictions

preds_full, targets_full, uncertainty_full = (
    collect_test_predictions(
        model,
        test_loader,
        device,
    )
)

# Reject 10% most uncertain ID samples

id_results = evaluate_id_after_pruning(
    preds_full,
    targets_full,
    uncertainty_full,
    prune_frac=0.10,
    num_classes=NUM_CLASSES,
    unk_label=UNK_LABEL,
)

# Reject 10% most uncertain samplesfrom complete test set

full_results = evaluate_full_pruning_composition(
    targets_full,
    uncertainty_full,
    prune_frac=0.10,
    unk_label=UNK_LABEL,
)